# 1. Project Title: SentimentScope - Linear SVM Model Training

This notebook trains a Linear Support Vector Machine (Linear SVM) classifier on the balanced dataset and evaluates its performance.

## 2. Import Libraries

In [2]:
import pandas as pd
import numpy as np
import os
import sys
import joblib
import time

sys.path.append("../src")
from text_normalizer import clean_text
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix


## 3. Load Dataset

In [3]:
CSV_PATH = "../data/balanced_reviews.csv"
MODEL_PATH = "../models/linear_svm_model.pkl"
VECTORIZER_PATH = "../models/tfidf_vectorizer_linear_svm.pkl"

print("Loading balanced dataset...")
df = pd.read_csv(CSV_PATH)
print(f"Shape: {df.shape}")

Loading balanced dataset...
Shape: (26400, 6)


## 4. Data Inspection

In [4]:
print("Columns in dataset:")
print(df.columns.tolist())

print("\nSentiment value counts:")
print(df['sentiment'].value_counts())

print("\nChecking for missing values:")
print(df[['review_text', 'sentiment']].isnull().sum())

Columns in dataset:
['product_id', 'product_title', 'category', 'review_text', 'rating', 'sentiment']

Sentiment value counts:
sentiment
Positive    8800
Neutral     8800
Negative    8800
Name: count, dtype: int64

Checking for missing values:
review_text    0
sentiment      0
dtype: int64


## 5. Data Cleaning

In [5]:
print("Cleaning missing values...")
df = df.dropna(subset=['review_text', 'sentiment'])
print(f"Shape after dropping nulls: {df.shape}")

print("\nNormalizing review text using clean_text()...")
df['clean_review'] = df['review_text'].astype(str).apply(clean_text)
print("Clean review text sample:")
print(df['clean_review'].iloc[0])

Cleaning missing values...
Shape after dropping nulls: (26400, 6)

Normalizing review text using clean_text()...
Clean review text sample:
great product provide good support while heavy lifting


## 6. Train/Test Split

In [6]:
print("Splitting dataset (80% train, 20% test)...")
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)
print(f"Training Samples: {X_train.shape[0]}")
print(f"Testing Samples:  {X_test.shape[0]}")

Splitting dataset (80% train, 20% test)...
Training Samples: 21120
Testing Samples:  5280


## 7. TF-IDF Vectorization

In [7]:
print("Fitting TF-IDF Vectorizer with Unigrams + Bigrams...")
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=15000, stop_words="english")

X_train_vectorized = vectorizer.fit_transform(X_train)
X_test_vectorized = vectorizer.transform(X_test)

print(f"Number of Features: {X_train_vectorized.shape[1]}")

Fitting TF-IDF Vectorizer with Unigrams + Bigrams...
Number of Features: 15000


## 8. Model Training

In [8]:
print("Training Linear SVM model...")
model = LinearSVC(random_state=42, max_iter=1000)

start_time = time.time()
model.fit(X_train_vectorized, y_train)
training_time = time.time() - start_time

print(f"Model training complete!")
print(f"Training Time: {training_time:.4f} seconds")

Training Linear SVM model...
Model training complete!
Training Time: 2.8612 seconds


## 9. Prediction

In [9]:
print("Making predictions on testing set...")
start_time = time.time()
y_pred = model.predict(X_test_vectorized)
prediction_time = time.time() - start_time

print(f"Prediction complete.")
print(f"Prediction Time: {prediction_time:.4f} seconds")

Making predictions on testing set...
Prediction complete.
Prediction Time: 0.0013 seconds


## 10. Evaluation Metrics

In [10]:
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print("==================================================")
print("EVALUATION METRICS")
print("==================================================")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

EVALUATION METRICS
Accuracy:  0.7684
Precision: 0.7677
Recall:    0.7684
F1 Score:  0.7677


## 11. Classification Report

In [11]:
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

Classification Report:

              precision    recall  f1-score   support

    Negative       0.77      0.76      0.76      1760
     Neutral       0.74      0.72      0.73      1760
    Positive       0.79      0.83      0.81      1760

    accuracy                           0.77      5280
   macro avg       0.77      0.77      0.77      5280
weighted avg       0.77      0.77      0.77      5280



## 12. Confusion Matrix

In [12]:
print("Confusion Matrix:\n")
print(confusion_matrix(y_test, y_pred, labels=["Negative", "Neutral", "Positive"]))

Confusion Matrix:

[[1330  253  177]
 [ 290 1260  210]
 [ 112  181 1467]]


## 13. Save Model

In [13]:
print(f"Saving trained model to: {MODEL_PATH}...")
joblib.dump(model, MODEL_PATH)
print("Model saved successfully!")

print(f"Saving TF-IDF vectorizer to: {VECTORIZER_PATH}...")
joblib.dump(vectorizer, VECTORIZER_PATH)
print("Vectorizer saved successfully!")

Saving trained model to: ../models/linear_svm_model.pkl...
Model saved successfully!
Saving TF-IDF vectorizer to: ../models/tfidf_vectorizer_linear_svm.pkl...
Vectorizer saved successfully!
